In [ ]:
%pip install langchain==0.2.11 langchain-openai langchain-community --quiet

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [ ]:
# CBT-LLM — Schema Memory Extraction (PER FILE)
# Input:  /data/processed/cleaned-gpt-conv/
# Output: /data/interim/user_schemas/

import json
import os
from pathlib import Path
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser


BASE_DIR = Path.cwd().resolve()
ROOT_DIR = BASE_DIR.parent

CLEAN_DIR = ROOT_DIR / "data" / "processed" / "cleaned-gpt-conv"
OUT_DIR = ROOT_DIR / "data" / "interim" / "user_schemas"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Reading cleaned files from:", CLEAN_DIR)
print("Saving schema memories to:", OUT_DIR)


llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,
)

parser = JsonOutputParser()

extract_prompt = ChatPromptTemplate.from_template("""
Extract CBT-related information from the user message.

Return valid JSON:

{{
  "trigger": "",
  "automatic_thoughts": [],
  "core_belief": [],
  "emotion": [],
  "behavior": ""
}}

USER:
{message}
""")

extract_chain = extract_prompt | llm | parser

clean_files = sorted(CLEAN_DIR.glob("*_cleaned.json"))
print(f"Found {len(clean_files)} cleaned conversation files.\n")

for file_path in clean_files:

    with open(file_path, "r") as f:
        turns = json.load(f)

    # create new schema memory for THIS conversation
    schema_memory = {
        "triggers": set(),
        "automatic_thoughts": set(),
        "core_beliefs": set(),
        "emotions": set(),
        "behaviors": set()
    }

    print(f"Processing {file_path.name} ...")

    for entry in turns:
        if entry["role"] != "user":
            continue

        msg = entry["content"].strip()
        if not msg:
            continue

        try:
            schema = extract_chain.invoke({"message": msg})

            # update schema memory
            if schema["trigger"]:
                schema_memory["triggers"].add(schema["trigger"])

            for x in schema["automatic_thoughts"]:
                schema_memory["automatic_thoughts"].add(x)

            for x in schema["core_belief"]:
                schema_memory["core_beliefs"].add(x)

            for x in schema["emotion"]:
                schema_memory["emotions"].add(x)

            if schema["behavior"]:
                schema_memory["behaviors"].add(schema["behavior"])

        except Exception as e:
            print("❌ Error in:", msg)
            print(e)

    # convert to JSON safe form
    schema_memory_json = {
        k: sorted(list(v)) for k, v in schema_memory.items()
    }

    # build output path
    out_path = OUT_DIR / f"{file_path.stem.replace('_cleaned','')}_schema_memory.json"

    with open(out_path, "w") as f:
        json.dump(schema_memory_json, f, indent=2)

    print(f"✔ Saved schema memory → {out_path.name}\n")

print("\n🎉 ALL DONE — One schema_memory file per conversation!")
